### Transformacion del folder "movie_languages"

In [0]:
%run "../../utilerias/funciones_comunes"

In [0]:
%run "../../utilerias/configurationes"

In [0]:
dbutils.widgets.text("param_file_date","2024-12-16")
v_file_date = dbutils.widgets.get("param_file_date")


#### Paso 1 - Leer los archivos json usando "DataFrameReader" de spark

In [0]:
#Importamos tipos de datos
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [0]:
#Definimos esquemas
schema_movie_languages = StructType([
    StructField('movieId', IntegerType(), True),
    StructField('languageId', IntegerType(), True),
     StructField('languageRoleId', IntegerType(), True)
])

In [0]:
#Leer los datos
movie_languages_df = spark.read.table("smartdata_proyect_bronze.movies_languages").filter(f"file_date = '{v_file_date}'") 


#### Paso 2 - Renombrar las columnas y añadir nuevas columnas
 1. "movieId" renombrar a "movie_id" 
 2. "languageId" renombrar a "language_id" 
 2. Eiminar columna "languageRoleId"
 3. Agregar las columnas "intestion_date" y "enviromet"

In [0]:
#Importaciones
from pyspark.sql.functions import col, current_timestamp, lit

In [0]:
movie_languages_final_df = movie_languages_df.withColumnsRenamed({"movieId": "movie_id","languageId": "language_id"})\
.withColumn("ingestion_date", current_timestamp())\
.drop('languageRoleId')    

#### Paso 3 - Escribir la salida en un formato "Delta Table"[](url) 

In [0]:
merge_condition = 'tgt.movie_id = src.movie_id AND tgt.language_id = src.language_id AND tgt.file_date = src.file_date'
merge_delta_lake("smartdata_proyect_silver","movies_languages",movie_languages_final_df, merge_condition, "file_date")

In [0]:
%sql
select count(1), file_date from smartdata_proyect_silver.movies_languages group by file_date;

In [0]:
dbutils.notebook.exit("success")